<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Copy_of_Volitility_regime_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# K-MEANS VOLATILITY REGIME CLUSTERING
# NIFTY 50 | SILHOUETTE ANALYSIS + K-MEANS
# ============================================================

!pip -q install yfinance scikit-learn scipy matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 1. DOWNLOAD NIFTY 50 DATA
# ------------------------------------------------------------

df = yf.download(
    "^NSEI",
    start="2015-01-01",
    end="2026-09-17",
    auto_adjust=True,
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# ------------------------------------------------------------
# 2. RETURN AND MARKET FEATURES
# ------------------------------------------------------------

df["Return"] = df["Close"].pct_change()

df["Log_Return"] = np.log(
    df["Close"] / df["Close"].shift(1)
)

# Absolute return — proxy for daily shock magnitude
df["Abs_Return"] = df["Return"].abs()

# Realized volatility
df["Volatility_5D"] = (
    df["Log_Return"]
    .rolling(5)
    .std() * np.sqrt(252)
)

df["Volatility_20D"] = (
    df["Log_Return"]
    .rolling(20)
    .std() * np.sqrt(252)
)

df["Volatility_60D"] = (
    df["Log_Return"]
    .rolling(60)
    .std() * np.sqrt(252)
)

# Volatility ratio
df["Vol_Ratio"] = (
    df["Volatility_5D"] /
    df["Volatility_60D"]
)

# Intraday high-low range
df["High_Low_Range"] = (
    (df["High"] - df["Low"]) /
    df["Close"]
)

# Volume relative to recent average
df["Volume_Change"] = (
    df["Volume"] /
    df["Volume"].rolling(20).mean()
    - 1
)

# Short-term momentum
df["Momentum_20D"] = (
    df["Close"] /
    df["Close"].shift(20)
    - 1
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print("=" * 75)
print("K-MEANS VOLATILITY REGIME CLUSTERING")
print("=" * 75)

print(f"Observations : {len(df):,}")

# ------------------------------------------------------------
# 3. FEATURE MATRIX
# ------------------------------------------------------------

features = [
    "Return",
    "Abs_Return",
    "Volatility_5D",
    "Volatility_20D",
    "Volatility_60D",
    "Vol_Ratio",
    "High_Low_Range",
    "Volume_Change",
    "Momentum_20D"
]

X = df[features].copy()

print("\nFeatures used:")
for feature in features:
    print("•", feature)

# ------------------------------------------------------------
# 4. STANDARDIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------
# 5. SILHOUETTE ANALYSIS
# ------------------------------------------------------------

k_values = range(2, 9)

inertias = []
silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(X_scaled)

    inertias.append(
        model.inertia_
    )

    silhouette_scores.append(
        silhouette_score(
            X_scaled,
            labels
        )
    )

evaluation = pd.DataFrame({

    "K": list(k_values),

    "Inertia": inertias,

    "Silhouette_Score":
        silhouette_scores
})

print("\nK-MEANS MODEL SELECTION")
print("-" * 75)

display(
    evaluation.style.format({

        "Inertia": "{:,.2f}",

        "Silhouette_Score":
            "{:.4f}"
    })
)

# ------------------------------------------------------------
# 6. SELECT K BASED ON SILHOUETTE SCORE
# ------------------------------------------------------------

best_k = int(
    evaluation.loc[
        evaluation[
            "Silhouette_Score"
        ].idxmax(),
        "K"
    ]
)

best_score = evaluation.loc[
    evaluation["K"] == best_k,
    "Silhouette_Score"
].iloc[0]

print(
    f"\nOptimal K              : {best_k}"
)

print(
    f"Best Silhouette Score  : {best_score:.4f}"
)

# ------------------------------------------------------------
# 7. ELBOW METHOD
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Inertia"],
    marker="o"
)

plt.axvline(
    best_k,
    linestyle="--",
    linewidth=2,
    label=f"Selected K = {best_k}"
)

plt.title(
    "K-Means Elbow Method — Volatility Regimes"
)

plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")

plt.xticks(
    list(k_values)
)

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 8. SILHOUETTE SCORE PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    evaluation["K"],
    evaluation["Silhouette_Score"],
    marker="o"
)

plt.axvline(
    best_k,
    linestyle="--",
    linewidth=2,
    label=f"Best K = {best_k}"
)

plt.title(
    "Silhouette Score — Volatility Regime Clustering"
)

plt.xlabel("Number of Clusters")

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(
    list(k_values)
)

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 9. FINAL K-MEANS MODEL
# ------------------------------------------------------------

final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=100
)

df["Cluster"] = (
    final_model.fit_predict(
        X_scaled
    )
)

# ------------------------------------------------------------
# 10. CLUSTER STATISTICS
# ------------------------------------------------------------

cluster_stats = df.groupby(
    "Cluster"
)[features].mean()

cluster_stats["Observations"] = (
    df["Cluster"]
    .value_counts()
    .sort_index()
)

cluster_stats["Frequency_%"] = (
    cluster_stats["Observations"] /
    len(df) * 100
)

print("\nCLUSTER CHARACTERISTICS")
print("-" * 75)

display(
    cluster_stats.style.format({

        "Return": "{:.4%}",
        "Abs_Return": "{:.4%}",

        "Volatility_5D": "{:.2%}",
        "Volatility_20D": "{:.2%}",
        "Volatility_60D": "{:.2%}",

        "Vol_Ratio": "{:.3f}",

        "High_Low_Range": "{:.2%}",

        "Volume_Change": "{:.2%}",

        "Momentum_20D": "{:.2%}",

        "Frequency_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# 11. IDENTIFY VOLATILITY ORDER
# ------------------------------------------------------------

vol_order = (
    cluster_stats[
        "Volatility_20D"
    ]
    .sort_values()
    .index
    .tolist()
)

regime_names = {}

if len(vol_order) == 2:

    regime_names = {
        vol_order[0]: "Low Volatility",
        vol_order[1]: "High Volatility"
    }

elif len(vol_order) == 3:

    regime_names = {
        vol_order[0]: "Low Volatility",
        vol_order[1]: "Medium Volatility",
        vol_order[2]: "High Volatility"
    }

else:

    for i, cluster in enumerate(vol_order):

        if i == 0:
            name = "Very Low Volatility"

        elif i == len(vol_order) - 1:
            name = "Very High Volatility"

        else:
            name = f"Volatility Regime {i+1}"

        regime_names[cluster] = name

df["Volatility_Regime"] = (
    df["Cluster"].map(regime_names)
)

print("\nCLUSTER → VOLATILITY REGIME")
print("-" * 75)

for cluster in sorted(regime_names):

    print(
        f"Cluster {cluster} → "
        f"{regime_names[cluster]}"
    )

# ------------------------------------------------------------
# 12. REGIME FREQUENCY
# ------------------------------------------------------------

regime_frequency = (
    df["Volatility_Regime"]
    .value_counts()
    .to_frame("Observations")
)

regime_frequency["Frequency_%"] = (
    regime_frequency["Observations"] /
    len(df) * 100
)

print("\nVOLATILITY REGIME FREQUENCY")
print("-" * 75)

display(
    regime_frequency.style.format({
        "Frequency_%": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# 13. REGIME RETURN & RISK PERFORMANCE
# ------------------------------------------------------------

regime_performance = df.groupby(
    "Volatility_Regime"
)["Return"].agg(
    Mean_Return="mean",
    Volatility="std",
    Positive_Days=lambda x: (
        x > 0
    ).mean()
)

regime_performance[
    "Annualized_Return"
] = (
    regime_performance["Mean_Return"]
    * 252
)

regime_performance[
    "Annualized_Volatility"
] = (
    regime_performance["Volatility"]
    * np.sqrt(252)
)

print("\nREGIME RETURN & RISK")
print("-" * 75)

display(
    regime_performance.style.format({

        "Mean_Return": "{:.4%}",

        "Volatility": "{:.4%}",

        "Positive_Days": "{:.2%}",

        "Annualized_Return":
            "{:.2%}",

        "Annualized_Volatility":
            "{:.2%}"
    })
)

# ------------------------------------------------------------
# 14. VOLATILITY REGIME TIMELINE
# ------------------------------------------------------------

regime_numeric = (
    df["Cluster"]
)

plt.figure(figsize=(16, 6))

plt.scatter(
    df.index,
    regime_numeric,
    c=regime_numeric,
    cmap="viridis",
    s=8
)

plt.title(
    "NIFTY 50 — K-Means Volatility Regimes"
)

plt.xlabel("Date")

plt.ylabel("Cluster")

plt.yticks(
    sorted(
        df["Cluster"].unique()
    )
)

plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 15. 20D VOLATILITY VS ABSOLUTE RETURN
# ------------------------------------------------------------

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df,
    x="Volatility_20D",
    y="Abs_Return",
    hue="Volatility_Regime",
    palette="viridis",
    alpha=0.6,
    s=35
)

plt.title(
    "Volatility Regimes — 20D Volatility vs Absolute Return"
)

plt.xlabel(
    "20-Day Annualized Volatility"
)

plt.ylabel(
    "Absolute Daily Return"
)

plt.grid(alpha=0.3)

plt.show()

# ------------------------------------------------------------
# 16. VOLATILITY DISTRIBUTION BY CLUSTER
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

sns.boxplot(
    data=df,
    x="Volatility_Regime",
    y="Volatility_20D"
)

plt.title(
    "20-Day Volatility Distribution Across Regimes"
)

plt.xlabel("Volatility Regime")

plt.ylabel(
    "20-Day Annualized Volatility"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()

# ------------------------------------------------------------
# 17. REGIME TRANSITION MATRIX
# ------------------------------------------------------------

transition_matrix = pd.crosstab(
    df["Volatility_Regime"],
    df["Volatility_Regime"].shift(-1),
    normalize="index"
)

print("\nVOLATILITY REGIME TRANSITION MATRIX")
print("-" * 75)

display(
    transition_matrix.round(4)
)

plt.figure(figsize=(10, 7))

sns.heatmap(
    transition_matrix,
    annot=True,
    fmt=".2f",
    cmap="viridis"
)

plt.title(
    "K-Means Volatility Regime Transition Matrix"
)

plt.xlabel("Next Regime")

plt.ylabel("Current Regime")

plt.tight_layout()

plt.show()

# ------------------------------------------------------------
# 18. REGIME DURATION
# ------------------------------------------------------------

regime_change = (
    df["Volatility_Regime"] !=
    df["Volatility_Regime"].shift(1)
)

episode_id = regime_change.cumsum()

duration_data = pd.DataFrame({

    "Regime":
        df["Volatility_Regime"].values,

    "Episode":
        episode_id.values
})

durations = (
    duration_data
    .groupby(
        ["Regime", "Episode"]
    )
    .size()
    .rename("Duration")
    .reset_index()
)

duration_summary = (
    durations
    .groupby("Regime")["Duration"]
    .agg(
        Episodes="count",
        Average_Duration="mean",
        Median_Duration="median",
        Maximum_Duration="max"
    )
)

print("\nVOLATILITY REGIME DURATION")
print("-" * 75)

display(
    duration_summary.round(2)
)

# ------------------------------------------------------------
# 19. CURRENT MARKET VOLATILITY STATE
# ------------------------------------------------------------

latest = df.iloc[-1]

print("\nCURRENT VOLATILITY STATE")
print("-" * 75)

print(
    f"Date              : "
    f"{df.index[-1].date()}"
)

print(
    f"Cluster           : "
    f"{int(latest['Cluster'])}"
)

print(
    f"Regime            : "
    f"{latest['Volatility_Regime']}"
)

print(
    f"20D Volatility    : "
    f"{latest['Volatility_20D']:.2%}"
)

print(
    f"60D Volatility    : "
    f"{latest['Volatility_60D']:.2%}"
)

print(
    f"Volatility Ratio  : "
    f"{latest['Vol_Ratio']:.3f}"
)

print(
    f"Daily Return      : "
    f"{latest['Return']:.2%}"
)

# ------------------------------------------------------------
# 20. FINAL SILHOUETTE VALIDATION
# ------------------------------------------------------------

final_silhouette = silhouette_score(
    X_scaled,
    df["Cluster"]
)

print("\nFINAL MODEL VALIDATION")
print("-" * 75)

print(
    f"Optimal K              : "
    f"{best_k}"
)

print(
    f"Final Silhouette Score : "
    f"{final_silhouette:.4f}"
)

# ------------------------------------------------------------
# 21. SAVE RESULTS
# ------------------------------------------------------------

df.to_csv(
    "NIFTY50_KMeans_Volatility_Regimes.csv"
)

evaluation.to_csv(
    "KMeans_Volatility_Silhouette_Evaluation.csv",
    index=False
)

cluster_stats.to_csv(
    "KMeans_Volatility_Cluster_Statistics.csv"
)

regime_performance.to_csv(
    "KMeans_Volatility_Regime_Performance.csv"
)

transition_matrix.to_csv(
    "KMeans_Volatility_Transition_Matrix.csv"
)

duration_summary.to_csv(
    "KMeans_Volatility_Regime_Duration.csv"
)

print("\nFiles saved:")
print("• NIFTY50_KMeans_Volatility_Regimes.csv")
print("• KMeans_Volatility_Silhouette_Evaluation.csv")
print("• KMeans_Volatility_Cluster_Statistics.csv")
print("• KMeans_Volatility_Regime_Performance.csv")
print("• KMeans_Volatility_Transition_Matrix.csv")
print("• KMeans_Volatility_Regime_Duration.csv")

print("\n" + "=" * 75)
print("PROJECT 4 COMPLETED")
print("=" * 75)